In [100]:
import re

In [101]:
def ler_arquivo(path):
    with open(path, "r", encoding='UTF-8') as file:
        return file.read()

In [102]:
contrato = ler_arquivo("contrato.md")
contrato

'# INSTRUMENTO PARTICULAR DE ACORDO EXTRAJUDICIAL E CONFISSÃO DE DÍVIDA\n\nPelo presente instrumento particular, de um lado, na qualidade de CREDOR, a Razão Social **SOLUCOES FINANCEIRAS ALPHA LTDA**,\npessoa jurídica de direito privado, inscrita regularmente no CNPJ sob o nº **12.345.678/0001-99**, com sede administrativa\nem São Paulo/SP, e de outro lado, na qualidade de DEVEDOR, o Sr. **CARLOS EDUARDO DOS SANTOS**, brasileiro, solteiro,\nportador da cédula de identidade RG nº 12.345.678-X e devidamente inscrito no CPF sob o nº **456.789.123-00**, residente\ne domiciliado na Rua das Oliveiras, nº 100, resolvem firmar o presente acordo mediante as seguintes cláusulas e condições:\n\n## CLÁUSULA PRIMEIRA:\nO DEVEDOR reconhece e confessa ser devedor da quantia líquida e certa de **R$ 12.450,50** relativa a débitos anteriores.\n\n## CLÁUSULA SEGUNDA:\nAs partes estabelecem que a data oficial de assinatura deste acordo é o dia **21/01/2026**, momento em que se torna válido para todos os e

In [103]:
def identificar_template(texto):
    # Divide sempre que encontrar \n## cláusula (case-insensitive)
    partes = re.split(r"(?i)(?=\n##\s+cláusula\b)", texto)
    # Descarta o primeiro bloco se ele não começar com "## cláusula" (o preâmbulo)
    clausulas = [p for p in partes if re.match(r"(?i)^##\s+cláusula\b", p.lstrip())]
    qtd = len(clausulas)

    if qtd == 13:
        tipo = "Modelo Simples"
    elif qtd == 16:
        tipo = "Modelo Completo"
    else:
        tipo = "Modelo Desconhecido"

    return tipo, qtd

In [104]:
identificar_template(contrato)


('Modelo Simples', 13)

In [105]:
def identificar_nome_cliente(texto):
    # Procura por "DEVEDOR" seguido de "o Sr." ou "a Sra." e depois o nome (com ou sem **)
    pattern = r'DEVEDOR,\s+(?:o Sr\.|a Sra\.)\s+\*?\*?([A-Za-z\s]+?)\*?\*?(?:,|\n)'
    match = re.search(pattern, texto, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    return None

nome_cliente = identificar_nome_cliente(contrato)
nome_cliente

'CARLOS EDUARDO DOS SANTOS'

In [106]:
def identificar_rg_cliente(texto):
    pattern = r"(?i)da\s+cédula\s+de\s+identidade\s+rg\s+n[º°o.]*\s*\**\s*([\d.\-]+(?:-[\dA-Za-z]|[\dA-Za-z])?)\s*\**"
    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

rg_cliente = identificar_rg_cliente(contrato)
rg_cliente

'12.345.678-X'

In [107]:
def identificar_cpf_cliente(texto):
    pattern = (
        r"(?i)inscrito\s+no\s+cpf\s+(?:sob\s+o\s+)?"
        r"n[º°o.]*\s*\**\s*(\d{3}\.\d{3}\.\d{3}-\d{2}|\d{11})\s*\**"
    )

    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

cpf_cliente = identificar_cpf_cliente(contrato)
cpf_cliente


'456.789.123-00'

In [108]:
def identificar_nome_empresa(texto):
    pattern = (
        r"(?i)(?:credor,\s*a\s+empresa|raz[aã]o\s+social\s*[:\-–—]?)\s*"
        r"\**([^*\r\n,]+?)\**\s*(?:,|\r?\n|$)"
    )

    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    return None

nome_empresa = identificar_nome_empresa(contrato)
nome_empresa

'SOLUCOES FINANCEIRAS ALPHA LTDA'

In [109]:
def identificar_cnpj_empresa(texto):
    pattern = (
        r"(?i)inscrita\s+regularmente\s+no\s+cnpj\s+(?:sob\s+o\s+)?"
        r"n[º°o.]*\s*\**\s*(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}|\d{14})\s*\**"
    )

    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

cnpj_empresa = identificar_cnpj_empresa(contrato)
cnpj_empresa 

'12.345.678/0001-99'

In [110]:
def identificar_valores_financeiros(texto, incluir_moeda=True):
    if incluir_moeda:
        pattern = r"\**\s*(R\$\s*(?:\d{1,3}(?:\.\d{3})*,\d{2}|\d+,\d{2}))\s*\**"
    else:
        pattern = r"\**\s*R\$\s*(\d{1,3}(?:\.\d{3})*,\d{2}|\d+,\d{2})\s*\**"

    return [re.sub(r"\s+", " ", v.strip()) for v in re.findall(pattern, texto)]

valores_financeiros = identificar_valores_financeiros(contrato)
valores_financeiros

['R$ 12.450,50', 'R$ 850,00', 'R$ 12.450,50', 'R$ 850,00', 'R$ 12.450,50']

In [111]:
def limpar_valor(valor_str):
    if not valor_str: return 0.0
    # Remove R$, espaços e pontos de milhar, substitui vírgula por ponto
    valor_limpo = valor_str.replace('R$', '').replace('.', '').replace(',', '.').strip()
    return float(valor_limpo)

valor = limpar_valor('R$800,01')
valor                     

800.01

In [112]:
def identificar_valor_total_divida(texto):
    pattern = r'quantia líquida e certa de\s+\*{0,2}R\$\s*([\d\.\,]+)\*{0,2}'
    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return limpar_valor(match.group(1).strip())

    return None

divida = identificar_valor_total_divida(contrato)
divida 

12450.5

In [113]:
def identificar_valor_divida_parcela_mensal(texto):
    pattern = r'(?i)(?:valor\s+de\s+)?cada\s+parcela(?:\s+mensal)?\s+'            r'(?:ser[aá]\s+(?:fixo\s+)?em|é\s+de|de)\s+'            r'\**\s*(R\$\s*(?:\d{1,3}(?:\.\d{3})*,\d{2}|\d+,\d{2}))\s*\**'
    
    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return limpar_valor(match.group(1).strip())

    return None

parcela = identificar_valor_divida_parcela_mensal(contrato)
parcela 

850.0

In [114]:
def identificar_data_assinatura(texto):
    pattern = (
        r"(?i)data\s+(?:oficial\s+)?de\s+assinatura"
        r"(?:\s+deste\s+(?:acordo|contrato|instrumento))?\s+"
        r"(?:[eé]\s+(?:o\s+dia\s+)?|:\s*)?\**\s*(\d{2}/\d{2}/\d{4})\s*\**"
    )
    match = re.search(pattern, texto)
    if match:
        return match.group(1).strip()

    return None

data_assinatura = identificar_data_assinatura(contrato)
data_assinatura

'21/01/2026'

In [115]:
def identificar_data_pagamento(texto):
    pattern = (
        r"(?i)(?:cr[eé]dito\s+em|"
        r"receber[aá]\s+o\s+dinheiro\s+em\s+sua\s+conta\s+banc[aá]ria\s+(?:[eé]\s+)?)"
        r"\s*\**\s*(\d{2}/\d{2}/\d{4})\s*\**"
    )
    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

data_pagamento = identificar_data_pagamento(contrato)
data_pagamento

'26/01/2026'

In [116]:
def identificar_data_primeira_parcela(texto):
    pattern = (
        r"(?i)primeira\s+parcela(?:\s+da\s+d[ií]vida)?\s+"
        r"(?:em|na\s+data\s+de|no\s+dia|vencimento\s+em|:)\s*"
        r"\**\s*(\d{2}/\d{2}/\d{4})\s*\**"
    )
    match = re.search(pattern, texto, flags=re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

primeira_parcela = identificar_data_primeira_parcela(contrato)
primeira_parcela

'10/03/2026'

In [124]:
template, qtd_cla = identificar_template(contrato)

dados = {
    "template": template,
    "clausulas_identificadas": qtd_cla,
    "cliente": {
        "nome": identificar_nome_cliente(contrato),
        "rg": identificar_rg_cliente(contrato),
        "cpf": identificar_cpf_cliente(contrato),        
    },
    "credor": {
        "nome": identificar_nome_empresa(contrato),
        "cnpj": identificar_cnpj_empresa(contrato),
    },
    "financeiro": {
        "valor_total": identificar_valor_total_divida(contrato),
        "valor_parcela": identificar_valor_divida_parcela_mensal(contrato),        
    },
    "datas": {
        "data_assinatura": identificar_data_assinatura(contrato),
        "data_pagamento" : identificar_data_pagamento(contrato),
        "data_vencimento_primeira_parcela": identificar_data_primeira_parcela(contrato),
    }
}

dados



{'template': 'Modelo Simples',
 'clausulas_identificadas': 13,
 'cliente': {'nome': 'CARLOS EDUARDO DOS SANTOS',
  'rg': '12.345.678-X',
  'cpf': '456.789.123-00'},
 'credor': {'nome': 'SOLUCOES FINANCEIRAS ALPHA LTDA',
  'cnpj': '12.345.678/0001-99'},
 'financeiro': {'valor_total': 12450.5, 'valor_parcela': 850.0},
 'datas': {'data_assinatura': '21/01/2026',
  'data_pagamento': '26/01/2026',
  'data_vencimento_primeira_parcela': '10/03/2026'}}

In [125]:
gabarito = {
  "template": "Modelo Simples",
  "clausulas_identificadas": 13,
  "cliente": {
    "nome": "CARLOS EDUARDO DOS SANTOS",
    "rg": "12.345.678-X",
    "cpf": "456.789.123-00"
  },
  "credor": {
    "nome": "SOLUCOES FINANCEIRAS ALPHA LTDA",
    "cnpj": "12.345.678/0001-99"
  },
  "financeiro": {
    "valor_total": 12450.50,
    "valor_parcela": 850.00
  },
  "datas": {
    "data_assinatura": "21/01/2026",
    "data_pagamento": "26/01/2026",
    "data_vencimento_primeira_parcela": "10/03/2026"
  }
}

In [126]:
assert dados == gabarito